# 04 - Modelling (KNN)
Notebook ini menjalankan preprocessing internal, split data train-test, KNN Regressor, dan GridSearchCV. Notebook dibuat standalone agar tidak bergantung pada folder output atau file perantara.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("baterai.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {DATA_PATH.resolve()}")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

EOL_THRESHOLD_RATIO = 0.80

def build_cycle_level_dataset(data: pd.DataFrame) -> pd.DataFrame:
    # Mengurutkan data agar agregasi dan perhitungan RUL konsisten.
    data = data.sort_values(["Battery_ID", "Cycle_Number", "Time_Seconds"]).copy()

    # Mengagregasi data sensor mentah menjadi data level siklus.
    cycle_df = (
        data.groupby(["Battery_ID", "Cycle_Number"], as_index=False)
        .agg(
            Time_Seconds_max=("Time_Seconds", "max"),
            Voltage_V_mean=("Voltage_V", "mean"),
            Voltage_V_std=("Voltage_V", "std"),
            Voltage_V_min=("Voltage_V", "min"),
            Voltage_V_max=("Voltage_V", "max"),
            Current_A_mean=("Current_A", "mean"),
            Current_A_std=("Current_A", "std"),
            Current_A_min=("Current_A", "min"),
            Current_A_max=("Current_A", "max"),
            Temperature_C_mean=("Temperature_C", "mean"),
            Temperature_C_std=("Temperature_C", "std"),
            Temperature_C_min=("Temperature_C", "min"),
            Temperature_C_max=("Temperature_C", "max"),
            Capacity_Ah=("Capacity_Ah", "mean"),
        )
        .sort_values(["Battery_ID", "Cycle_Number"])
    )

    # Standar deviasi dapat NaN jika satu siklus hanya memiliki satu observasi.
    std_columns = [col for col in cycle_df.columns if col.endswith("_std")]
    cycle_df[std_columns] = cycle_df[std_columns].fillna(0)

    # Membentuk indikator State of Health (SOH).
    cycle_df["Initial_Capacity_Ah"] = cycle_df.groupby("Battery_ID")["Capacity_Ah"].transform("first")
    cycle_df["SOH_Percent"] = cycle_df["Capacity_Ah"] / cycle_df["Initial_Capacity_Ah"] * 100

    # Membentuk target RUL_Cycles berdasarkan sisa siklus menuju End-of-Life.
    processed_parts = []
    for battery_id, part in cycle_df.groupby("Battery_ID", sort=False):
        part = part.sort_values("Cycle_Number").copy()
        initial_capacity = part["Initial_Capacity_Ah"].iloc[0]
        eol_capacity = EOL_THRESHOLD_RATIO * initial_capacity
        below_eol = part.loc[part["Capacity_Ah"] <= eol_capacity, "Cycle_Number"]
        eol_cycle = below_eol.iloc[0] if not below_eol.empty else part["Cycle_Number"].max()

        part["EOL_Cycle"] = eol_cycle
        part["RUL_Cycles"] = (eol_cycle - part["Cycle_Number"]).clip(lower=0)
        part["Normalized_Cycle"] = (
            (part["Cycle_Number"] - part["Cycle_Number"].min())
            / max(part["Cycle_Number"].max() - part["Cycle_Number"].min(), 1)
        )
        processed_parts.append(part)

    return pd.concat(processed_parts, ignore_index=True)

# Membentuk dataset level siklus dan memisahkan fitur-target.
df = pd.read_csv(DATA_PATH)
cycle_df = build_cycle_level_dataset(df)
TARGET_COLUMN = "RUL_Cycles"
X = cycle_df.drop(columns=["Battery_ID", "EOL_Cycle", TARGET_COLUMN]).select_dtypes(include="number")
y = cycle_df[TARGET_COLUMN]

# Imputasi dan scaling preview untuk menunjukkan hasil preprocessing.
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
scaler_preview = StandardScaler()
X_scaled_preview = pd.DataFrame(scaler_preview.fit_transform(X_imputed), columns=X.columns)

print(f"Ukuran data level siklus: {cycle_df.shape[0]:,} baris dan {cycle_df.shape[1]:,} kolom")
print(f"Jumlah fitur numerik: {X.shape[1]}")
print(f"Target: {TARGET_COLUMN}")
display(cycle_df.head())
display(X.isna().sum().to_frame("Missing Value Fitur"))
display(X_scaled_preview.head())

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Membagi data menjadi train dan test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Pipeline mencegah data leakage karena scaling dipelajari hanya dari data latih.
pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor()),
    ]
)

# Grid search sederhana untuk mencari K terbaik.
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=1,  # Lebih stabil untuk eksekusi notebook di laptop/VS Code
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f"Jumlah data latih: {len(X_train):,}")
print(f"Jumlah data uji: {len(X_test):,}")
print("Parameter terbaik:", grid_search.best_params_)
print(f"Skor CV terbaik (RMSE): {-grid_search.best_score_:.4f}")

Ukuran data level siklus: 2,769 baris dan 21 kolom
Jumlah fitur numerik: 18
Target: RUL_Cycles


,Battery_ID,Cycle_Number,Time_Seconds_max,Voltage_V_mean,Voltage_V_std,Voltage_V_min,Voltage_V_max,Current_A_mean,Current_A_std,Current_A_min,Current_A_max,Temperature_C_mean,Temperature_C_std,Temperature_C_min,Temperature_C_max,Capacity_Ah,Initial_Capacity_Ah,SOH_Percent,EOL_Cycle,RUL_Cycles,Normalized_Cycle
0,B0005,2,3690.234,3.529829,0.236558,2.612467,4.191492,-1.818702,0.595058,-2.018015,0.000729,32.572328,3.495804,24.325993,38.982181,1.856487,1.856487,100.000000,356,354,0.000000
1,B0005,4,3672.344,3.537320,0.235366,2.587209,4.189773,-1.817560,0.596704,-2.016821,0.002927,32.725235,3.435509,24.685948,39.033398,1.846327,1.856487,99.452721,356,352,0.003268
2,B0005,6,3651.641,3.543737,0.228111,2.651917,4.188187,-1.816487,0.598033,-2.016574,0.001484,32.642862,3.388174,24.734266,38.818797,1.835349,1.856487,98.861386,356,350,0.006536
3,B0005,8,3631.563,3.543666,0.233347,2.592948,4.188461,-1.825589,0.584972,-2.015936,0.001547,32.514876,3.395306,24.652244,38.762305,1.835263,1.856487,98.856718,356,348,0.009804
4,B0005,10,3629.172,3.542343,0.237301,2.547420,4.188299,-1.826114,0.584978,-2.017426,0.001701,32.382349,3.404667,24.518700,38.665393,1.834646,1.856487,98.823482,356,346,0.013072


,Missing Value Fitur
Cycle_Number,0
Time_Seconds_max,0
Voltage_V_mean,0
Voltage_V_std,0
Voltage_V_min,0
Voltage_V_max,0
Current_A_mean,0
Current_A_std,0
Current_A_min,0
Current_A_max,0


,Cycle_Number,Time_Seconds_max,Voltage_V_mean,Voltage_V_std,Voltage_V_min,Voltage_V_max,Current_A_mean,Current_A_std,Current_A_min,Current_A_max,Temperature_C_mean,Temperature_C_std,Temperature_C_min,Temperature_C_max,Capacity_Ah,Initial_Capacity_Ah,SOH_Percent,Normalized_Cycle
0,-1.136729,0.333008,0.496674,-0.779521,0.942219,0.160669,-0.115092,-0.270747,0.401821,-0.207900,0.404338,-0.157804,0.410153,0.276479,1.121738,0.952745,-0.323278,-1.687920
1,-1.122792,0.320123,0.534430,-0.793749,0.860449,0.114758,-0.113630,-0.268149,0.402902,0.667248,0.415357,-0.184535,0.439876,0.279616,1.100232,0.952745,-0.324273,-1.676818
2,-1.108855,0.305213,0.566769,-0.880338,1.069929,0.072376,-0.112256,-0.266049,0.403126,0.092633,0.409421,-0.205521,0.443866,0.266475,1.076994,0.952745,-0.325347,-1.665716
3,-1.094918,0.290753,0.566413,-0.817853,0.879029,0.079706,-0.123906,-0.286676,0.403704,0.117841,0.400198,-0.202359,0.437093,0.263016,1.076811,0.952745,-0.325356,-1.654614
4,-1.080981,0.289031,0.559746,-0.770657,0.731640,0.075362,-0.124578,-0.286667,0.402355,0.179310,0.390648,-0.198209,0.426066,0.257081,1.075505,0.952745,-0.325416,-1.643512


Fitting 5 folds for each of 32 candidates, totalling 160 fits


Jumlah data latih: 2,215
Jumlah data uji: 554
Parameter terbaik: {'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}
Skor CV terbaik (RMSE): 14.2302


In [2]:
display(Markdown(f"""
**Analisis Hasil**

Model yang digunakan adalah **K-Nearest Neighbors Regressor**. KNN memprediksi nilai kontinu berdasarkan kedekatan suatu data terhadap sejumlah tetangga terdekatnya. Data dibagi menjadi data latih dan data uji dengan rasio 80:20. GridSearchCV digunakan untuk mencari kombinasi parameter terbaik, meliputi jumlah tetangga, jenis pembobotan, dan metrik jarak. Parameter terbaik yang diperoleh adalah `{grid_search.best_params_}` dengan skor validasi silang RMSE sebesar `{-grid_search.best_score_:.4f}`.
"""))


**Analisis Hasil**

Model yang digunakan adalah **K-Nearest Neighbors Regressor**. KNN memprediksi nilai kontinu berdasarkan kedekatan suatu data terhadap sejumlah tetangga terdekatnya. Data dibagi menjadi data latih dan data uji dengan rasio 80:20. GridSearchCV digunakan untuk mencari kombinasi parameter terbaik, meliputi jumlah tetangga, jenis pembobotan, dan metrik jarak. Parameter terbaik yang diperoleh adalah `{'knn__metric': 'manhattan', 'knn__n_neighbors': 3, 'knn__weights': 'distance'}` dengan skor validasi silang RMSE sebesar `14.2302`.
